# Metafeatures Relevance Testing

In [1]:
%pip install --upgrade jupyter ipywidgets --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
DATABASE_PATH = "/mnt/sata1/lam/EMBER2024/NEW.sqlite3"
NUM_FEATURES = 2568

In [3]:
import sqlite3
import numpy as np

class MyDataLoader:
    def __init__(self) -> None:
        self.conn = sqlite3.connect(DATABASE_PATH)
        self.cursor = self.conn.cursor()
        self.cursor.execute("SELECT sha256, label, feature_vector FROM files ORDER BY sha256;")

    def load_next(self, count: int) -> tuple[np.ndarray, np.ndarray]:
        """Returns X and y for the NEXT `count` samples from the database."""

        X = []
        y = []
        for row in self.cursor.fetchmany(count):
            sha256, label, feature_vector = row
            # print(f"SHA256: {sha256}, Label: {label}, Feature Vector Length: {len(feature_vector)}")
            feature_vector = np.frombuffer(feature_vector, dtype=np.float32)
            X0 = feature_vector
            y0 = label
            X.append(X0)
            y.append(y0)
        
        return np.array(X), np.array(y)

loader = MyDataLoader()

In [4]:
X_train, y_train = loader.load_next(131072)
X_val, y_val = loader.load_next(16384)
X_test, y_test = loader.load_next(16384)

In [18]:
from utils.metafeature_experiment import MetafeatureExperiment
from utils.feature_vector_transformer import FeatureAddition, FeatureRemoval

In [6]:
def run_experiment(alterations: list):
    experiment = MetafeatureExperiment(
        alterations=alterations
    )
    result = experiment.run(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
    )
    print(str(result))
    return result


In [7]:

baseline_result = run_experiment(alterations=[])

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999846559613
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [8]:


experiment1_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',

            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;pointer_to_symbol_table', #
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
            'HeaderFileInfo;OPTIONAL;address_of_entrypoint', #
            'HeaderFileInfo;OPTIONAL;base_of_code', #
            'HeaderFileInfo;OPTIONAL;image_base', #
        ]),
    ]
)


[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999742905021
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [9]:
experiment2_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
        ]),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999778698341
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [10]:
experiment3_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
            # 'HeaderFileInfo;COFF;pointer_to_symbol_table',
            # 'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            # 'HeaderFileInfo;OPTIONAL;base_of_code',
            # 'HeaderFileInfo;OPTIONAL;image_base',
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999750609079
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [11]:
experiment4_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;pointer_to_symbol_table',
            'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            'HeaderFileInfo;OPTIONAL;base_of_code',
            'HeaderFileInfo;OPTIONAL;image_base',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999807324055
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [12]:
experiment5_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;pointer_to_symbol_table',
            'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            'HeaderFileInfo;OPTIONAL;base_of_code',
            'HeaderFileInfo;OPTIONAL;image_base',
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999757761782
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [13]:
experiment6_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'GeneralFileInfo;size',
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999838870457
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [14]:
experiment7_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            # from experiment 6
            'GeneralFileInfo;size',
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',

            # from experiment 3
            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
        ]),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999714145193
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [15]:
experiment8_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            # from experiment 6
            'GeneralFileInfo;size',
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',

            # from experiment 2
            'HeaderFileInfo;COFF;timestamp',
        ]),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999797146355
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [16]:
[r.auc for r in [
    baseline_result,
    experiment1_result,
    experiment2_result,
    experiment3_result,
    experiment4_result,
    experiment5_result,
    experiment6_result,
    experiment7_result,
    experiment8_result,
]]

[0.9998465596130506,
 0.9997429050212648,
 0.9997786983406249,
 0.9997506090787623,
 0.9998073240552341,
 0.9997577617820483,
 0.9998388704570182,
 0.9997141451934691,
 0.9997971463545168]

In [17]:
np.array([r.auc for r in [
    baseline_result,
    experiment1_result,
    experiment2_result,
    experiment3_result,
    experiment4_result,
    experiment5_result,
    experiment6_result,
    experiment7_result,
    experiment8_result,
]]) > np.array([baseline_result.auc])

array([False, False, False, False, False, False, False, False, False])

In [20]:
from utils.feature_vector_transformer import FeatureVectorView

class SizeofCodePerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofCodePerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_code = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_code')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_code / sizeof_image


In [21]:
class SizeofInitializedDataPerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofInitializedDataPerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_initialized_data = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_initialized_data')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_initialized_data / sizeof_image

In [22]:
class SizeofUninitializedDataPerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofUninitializedDataPerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_uninitialized_data = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_uninitialized_data')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_uninitialized_data / sizeof_image

In [24]:
class SizeofHeadersPerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofHeadersPerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_headers = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_headers')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_headers / sizeof_image

In [25]:
class MaxSectionVsizeAndSizeRatiosDelta(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("MaxSectionVsizeAndSizeRatiosDelta")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        max_vsize = vector_view.get('SectionInfo;general;section_vsize_ratio;max')
        max_size = vector_view.get('SectionInfo;general;section_size_ratio;max')
        return max_vsize - max_size


In [ ]:
experiment9_result = run_experiment(
    alterations=[
        SizeofCodePerSizeofImage(),
        SizeofInitializedDataPerSizeofImage(),
        SizeofUninitializedDataPerSizeofImage(),
        SizeofHeadersPerSizeofImage(),
        MaxSectionVsizeAndSizeRatiosDelta(),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.999826383029
New Feature Impacts:
  - SizeofCodePerSizeofImage: E(|SHAP|)=0.008808, Ranking=145, Percentile=94.36%, Delta from Avg=0.003149, Delta from Max=-1.878101, Delta from Min=0.008808
  - SizeofInitializedDataPerSizeofImage: E(|SHAP|)=0.010911, Ranking=119, Percentile=95.38%, Delta from Avg=0.005253, Delta from Max=-1.875998, Delta from Min=0.010911
  - SizeofUninitializedDataPerSizeofImage: E(|SHAP|)=0.000631, Ranking=882, Percen

/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


: 

# Good bye